<a href="https://colab.research.google.com/github/jairaj023/Internship-Practice/blob/main/Day_13Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import libraries

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Import dataset

In [2]:
from google.colab import files
upload = files.upload()

Saving raw_jobs.csv to raw_jobs.csv


In [3]:
df = pd.read_csv("raw_jobs.csv")

Check missing values

In [26]:
print(df.isnull().sum())

job_id             0
job_title          0
company            0
location           0
job_description    0
experience         0
education          0
salary             0
job_type           0
dtype: int64


In [4]:
print("Shape:", df.shape)

Shape: (10000, 9)


In [5]:
print("\nColumns:")
print(df.columns.tolist())


Columns:
['job_id', 'job_title', 'company', 'location', 'job_description', 'experience', 'education', 'salary', 'job_type']


In [6]:
print("\nFirst 5 rows:")
display(df.head())


First 5 rows:


,job_id,job_title,company,location,job_description,experience,education,salary,job_type
0,JOB00001,Python Developer,Infosys,Remote,We are looking for a Python Developer to join ...,1-3 years,MCA,₹10-16 LPA,Part-time
1,JOB00002,Data Scientist,Infosys,"Hyderabad, Telangana",We are looking for a Data Scientist to join ou...,0-2 years,MCA,₹12-20 LPA,Remote
2,JOB00003,Data Engineer,IBM,"Bengaluru, Karnataka",We are looking for a Data Engineer to join our...,8-12 years,BCA,₹3-5 LPA,Internship
3,JOB00004,Business Analyst,Google,"Hyderabad, Telangana",We are looking for a Business Analyst to join ...,1-3 years,M.Tech,₹10-16 LPA,Remote
4,JOB00005,Business Analyst,Mphasis,"Delhi, India",We are looking for a Business Analyst to join ...,8-12 years,M.Tech,₹5-8 LPA,Remote


Create labels from job descriptions

In [27]:
def assign_category(text):
    text = str(text).lower()

    # Programming
    if any(skill in text for skill in [
        "python", "java", "c++", "javascript", "c#", "ruby"
    ]):
        return "PROGRAMMING"

    # Database
    elif any(skill in text for skill in [
        "sql", "mysql", "postgresql", "mongodb", "oracle", "database"
    ]):
        return "DATABASE"

    # Cloud
    elif any(skill in text for skill in [
        "aws", "azure", "google cloud", "gcp", "cloud computing"
    ]):
        return "CLOUD"

    # BI Tools
    elif any(skill in text for skill in [
        "power bi", "powerbi", "tableau", "looker", "business intelligence"
    ]):
        return "BI_TOOL"

    # Data Science / ML
    elif any(skill in text for skill in [
        "machine learning", "data science", "tensorflow",
        "pytorch", "scikit-learn", "pandas", "numpy"
    ]):
        return "DATA_SCIENCE"

    # Web Development
    elif any(skill in text for skill in [
        "html", "css", "react", "angular", "node.js",
        "django", "flask", "frontend", "backend"
    ]):
        return "WEB_DEVELOPMENT"

    # DevOps
    elif any(skill in text for skill in [
        "docker", "kubernetes", "jenkins", "devops", "ci/cd"
    ]):
        return "DEVOPS"

    else:
        return "OTHER"

Apply the labels

In [28]:
df["category"] = df["job_description"].apply(assign_category)

display(df[["job_title", "job_description", "category"]].head(10))

,job_title,job_description,category
0,Python Developer,We are looking for a Python Developer to join ...,PROGRAMMING
1,Data Scientist,We are looking for a Data Scientist to join ou...,PROGRAMMING
2,Data Engineer,We are looking for a Data Engineer to join our...,PROGRAMMING
3,Business Analyst,We are looking for a Business Analyst to join ...,DATABASE
4,Business Analyst,We are looking for a Business Analyst to join ...,DATABASE
5,Software Engineer,We are looking for a Software Engineer to join...,PROGRAMMING
6,Frontend Developer,We are looking for a Frontend Developer to joi...,PROGRAMMING
7,AI Engineer,We are looking for a AI Engineer to join our t...,DATA_SCIENCE
8,Frontend Developer,We are looking for a Frontend Developer to joi...,WEB_DEVELOPMENT
9,AI Engineer,We are looking for a AI Engineer to join our t...,PROGRAMMING


Check category distribution

In [29]:
print(df["category"].value_counts())

category
PROGRAMMING        6045
DATABASE           2277
CLOUD               830
WEB_DEVELOPMENT     317
BI_TOOL             186
DATA_SCIENCE        126
OTHER               125
DEVOPS               94
Name: count, dtype: int64


Prepare X and y

In [30]:
X = df["job_description"].fillna("").astype(str)
y = df["category"]

Remove very small categories

In [31]:
category_counts = y.value_counts()

valid_categories = category_counts[category_counts >= 2].index

df_model = df[df["category"].isin(valid_categories)].copy()

X = df_model["job_description"].fillna("").astype(str)
y = df_model["category"]

print("Final dataset shape:", df_model.shape)
print("\nCategory distribution:")
print(y.value_counts())

Final dataset shape: (10000, 10)

Category distribution:
category
PROGRAMMING        6045
DATABASE           2277
CLOUD               830
WEB_DEVELOPMENT     317
BI_TOOL             186
DATA_SCIENCE        126
OTHER               125
DEVOPS               94
Name: count, dtype: int64


Split training and testing data

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 8000
Testing data: 2000


Create TF-IDF + Logistic Regression Pipeline

In [33]:
model = Pipeline([("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))), ("classifier", LogisticRegression(max_iter=1000))])

Train the model

In [35]:
model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


Predict

In [37]:
y_pred = model.predict(X_test)

print("Predictions:")
print(y_pred[:20])

Predictions:
['PROGRAMMING' 'PROGRAMMING' 'DATABASE' 'PROGRAMMING' 'DATABASE'
 'DATABASE' 'DATABASE' 'PROGRAMMING' 'DATABASE' 'PROGRAMMING' 'DATABASE'
 'PROGRAMMING' 'PROGRAMMING' 'PROGRAMMING' 'PROGRAMMING' 'PROGRAMMING'
 'PROGRAMMING' 'PROGRAMMING' 'PROGRAMMING' 'DATABASE']


Check accuracy

In [38]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", round(accuracy * 100, 2), "%")

Accuracy: 0.9865
Accuracy Percentage: 98.65 %


Classification report

In [39]:
print(classification_report(y_test, y_pred, zero_division=0))

                 precision    recall  f1-score   support

        BI_TOOL       1.00      0.92      0.96        37
          CLOUD       0.98      0.99      0.98       166
       DATABASE       0.98      0.98      0.98       455
   DATA_SCIENCE       1.00      0.76      0.86        25
         DEVOPS       1.00      0.95      0.97        19
          OTHER       1.00      0.80      0.89        25
    PROGRAMMING       0.99      1.00      0.99      1209
WEB_DEVELOPMENT       0.97      1.00      0.98        64

       accuracy                           0.99      2000
      macro avg       0.99      0.92      0.95      2000
   weighted avg       0.99      0.99      0.99      2000



Test with a new job description

In [40]:
new_job = ["We are looking for a Python developer with experience in Django and REST APIs."]

prediction = model.predict(new_job)

print("Job Description:")
print(new_job[0])

print("\nPredicted Category:")
print(prediction[0])

Job Description:
We are looking for a Python developer with experience in Django and REST APIs.

Predicted Category:
PROGRAMMING


Test multiple descriptions

In [44]:
new_jobs = [
    "Experience with Python and Java programming",
    "Strong SQL and MySQL database knowledge",
    "Experience working with AWS and Azure",
    "Create dashboards using Power BI and Tableau",
    "Machine learning experience with Python and TensorFlow",
    "React and JavaScript frontend development",
    "Docker and Kubernetes experience"
]

predictions = model.predict(new_jobs)

for job, category in zip(new_jobs, predictions):
    print("Job:", job)
    print("Category:", category)
    print("\n")

Job: Experience with Python and Java programming
Category: PROGRAMMING


Job: Strong SQL and MySQL database knowledge
Category: DATABASE


Job: Experience working with AWS and Azure
Category: CLOUD


Job: Create dashboards using Power BI and Tableau
Category: PROGRAMMING


Job: Machine learning experience with Python and TensorFlow
Category: PROGRAMMING


Job: React and JavaScript frontend development
Category: PROGRAMMING


Job: Docker and Kubernetes experience
Category: PROGRAMMING




Save the model

In [45]:
joblib.dump(model, "skill_classifier.pkl")

print("Model saved successfully!")
print("File: skill_classifier.pkl")

Model saved successfully!
File: skill_classifier.pkl


Verify the file

In [46]:
import os

print(os.path.exists("skill_classifier.pkl"))

True


Load the saved model

In [47]:
loaded_model = joblib.load("skill_classifier.pkl")

print("Model loaded successfully!")

Model loaded successfully!


Test the saved model

In [48]:
test_job = ["Looking for a data analyst with SQL, Python and Power BI experience."]

result = loaded_model.predict(test_job)

print("Input:", test_job[0])
print("Predicted Category:", result[0])

Input: Looking for a data analyst with SQL, Python and Power BI experience.
Predicted Category: PROGRAMMING


Download skill_classifier.pkl

In [49]:
files.download("skill_classifier.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>